In [ ]:
# Scenario Question
# "Imagine you are designing an AI-powered assistant that drafts structured reports, pauses for human review, and then either publishes or rejects them. How would you structure the state and workflow graph to ensure accountability and human oversight in the process?"

In [3]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, List, Dict
import requests
import os
from dotenv import load_dotenv
from datetime import datetime

load_dotenv()

# ================= STATE =================
class ReportState(TypedDict):
    task: str
    draft: str
    approved: bool
    reviewer: str
    feedback: str
    status: str
    history: List[Dict]


# ================= NVIDIA API =================
def nvidia_call(prompt: str):
    api_key = os.getenv("NVIDIA_API_KEY")

    response = requests.post(
        "https://integrate.api.nvidia.com/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json"
        },
        json={
            "model": "meta/llama3-8b-instruct",
            "messages": [
                {"role": "system", "content": "You are a professional report assistant."},
                {"role": "user", "content": prompt}
            ],
            "temperature": 0.5
        }
    )

    if response.status_code != 200:
        raise Exception(response.text)

    return response.json()["choices"][0]["message"]["content"]


# ================= NODES =================

# Draft Node
def draft_report(state: ReportState):
    print(f"\n📝 Generating report for: {state['task']}")

    draft = nvidia_call(f"""
Create a structured report for: {state['task']}

Include:
- Title
- Introduction
- Key Points
- Conclusion
""")

    return {
        "draft": draft,
        "status": "DRAFTED"
    }


# Review Node (Interrupt)
def review_node(state: ReportState):
    print("\n📄 Draft for Review:\n")
    print(state["draft"])
    return {}


# Decision Node
def decision_node(state: ReportState):
    return "publish" if state["approved"] else "reject"


# Publish Node
def publish_report(state: ReportState):
    print("\n✅ Report Published")

    entry = {
        "action": "PUBLISHED",
        "reviewer": state["reviewer"],
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }

    return {
        "status": "PUBLISHED",
        "history": state["history"] + [entry]
    }


# Reject Node
def reject_report(state: ReportState):
    print("\n❌ Report Rejected")

    entry = {
        "action": "REJECTED",
        "reviewer": state["reviewer"],
        "feedback": state["feedback"],
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }

    return {
        "status": "REJECTED",
        "history": state["history"] + [entry]
    }


# ================= GRAPH =================
graph = StateGraph(ReportState)

graph.add_node("draft", draft_report)
graph.add_node("review", review_node)
graph.add_node("decision", decision_node)
graph.add_node("publish", publish_report)
graph.add_node("reject", reject_report)

graph.set_entry_point("draft")

graph.add_edge("draft", "review")
graph.add_edge("review", "decision")

graph.add_conditional_edges(
    "decision",
    lambda state: "publish" if state["approved"] else "reject",
    {
        "publish": "publish",
        "reject": "reject"
    }
)

graph.add_edge("publish", END)
graph.add_edge("reject", END)


# ================= COMPILE =================
checkpointer = MemorySaver()

app = graph.compile(
    checkpointer=checkpointer,
    interrupt_before=["review"]
)


# ================= RUN =================
if __name__ == "__main__":
    thread = {"configurable": {"thread_id": "report-1"}}

    # Step 1: Generate Draft (ONLY ONCE)
    app.invoke({
        "task": "Q3 Financial Report",
        "draft": "",
        "approved": False,
        "reviewer": "",
        "feedback": "",
        "status": "",
        "history": []
    }, thread)

    # Step 2: Human Review
    decision = input("\nApprove? (yes/no): ").strip().lower()
    feedback = input("Feedback: ")
    reviewer = input("Reviewer: ")

    approved = True if decision == "yes" else False

    # Step 3: Resume (NO duplicate draft now)
    result = app.invoke({
        "approved": approved,
        "feedback": feedback,
        "reviewer": reviewer
    }, thread)

    print("\n📊 FINAL STATE:")
    print(result)


📝 Generating report for: Q3 Financial Report

📝 Generating report for: Q3 Financial Report

📊 FINAL STATE:
{'task': 'Q3 Financial Report', 'draft': "**Q3 Financial Report**\n\n**Title:** Quarterly Financial Report for the Period Ending September 30, 2022\n\n**Introduction:**\n\nThis report provides an overview of the company's financial performance for the third quarter of 2022 (Q3). The report covers the period from July 1, 2022, to September 30, 2022, and provides an analysis of the company's financial position, results of operations, and cash flows.\n\n**Key Points:**\n\n**Financial Position:**\n\n* Total Assets: $1,234,567 (up 5% from Q2 2022)\n* Total Liabilities: $543,210 (down 3% from Q2 2022)\n* Equity: $691,357 (up 8% from Q2 2022)\n* Current Ratio: 1.23 (improved from 1.15 in Q2 2022)\n\n**Results of Operations:**\n\n* Revenue: $1,234,567 (up 10% from Q2 2022)\n* Gross Profit: $543,210 (up 12% from Q2 2022)\n* Operating Expenses: $391,019 (up 5% from Q2 2022)\n* Net Income: 